## Initialization and configuration

In [9]:
%load_ext autoreload
%autoreload 2
import os
import sys

# Tell the OpenMP backend to ignore duplicate initializations instead of panicking
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# Force single-threaded execution for baseline math libraries to avoid runtime collisions
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Now import your libraries safely
import torch
import numpy as np
import pyro

In [10]:
#importing necessaryn modules and functions
import os
import sys
import torch
import pyro
import pyro.infer as infer
import arviz as az
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
sys.path.append(os.path.abspath(".."))
from src.model import PhylogeneticPrior
from src.penalty import get_fresh_test_quartets
from src.diagnostics import evaluate_test_diagnostics

#geometry parameters
N = 50 #number of points
K = 2 #number of dimensions
sigma_u = 1.0 #scale of normal base prior on latent vectors

#quartet data subsampling budgets
B_train = 3000 #subset of quartets usedto create landscape and MCMC potential gradients
B_test = 1000 #fresh quartets for evaluation 

#hyperparameters 
tau = 0.02 #relaxation temperature
lambda_grid = [0.0, 0.01, 0.1, 1.0, 10.0, 25.0] #lambda represents penalty strength
#we sweep across increasing values to evaluate the effect of penalty strength

#MCMC settings
n_samples = 400 #no of posterior draws per chain
n_warmup = 200 #discarded adaptation steps per chain
n_chains = 2 #indepedent markov chains

#initializing generative model space
model_instance = PhylogeneticPrior(N=N, K=K, B=B_train, seed = 42)

#extracting fresh validation quartets
test_quartets = get_fresh_test_quartets(N= N, B_test = B_test, train_quartets = model_instance.fixed_indices, seed = 123)

## Sampling engine
Running NUTS sampler and storing raw distance matrices for all configurations (posterior samples) into dictionary named experiment_results.

In [12]:
# storage for downstream analysis
posterior_distance_matrices = {}
sampler_health_records = {}

print(f"LAUNCHING HYPERBOLIC PRIOR SWEEP (taxa : {N}, Penalty budget: {B_train})")

for lmbda in lambda_grid:
    print(f"\n Sampling active landscape for lambda = {lmbda}")
    chain_samples = []
    chain_diagnostics = [] 
    chain_D_matrices = [] 

    for chain_idx in range(n_chains):
        print(f" Executing markov chain {chain_idx + 1} / {n_chains}..")
        pyro.clear_param_store()
        pyro.set_rng_seed(42 + chain_idx)

        # Re-instantiate the class locally for a clean trace graph
        local_model = PhylogeneticPrior(N=N, K=K, B=B_train, seed=42)

        def target_potential():
            # FIXED: Changed keyword argument 'lmbda' to 'lmbda_4' to match src/model.py
            return local_model.initialize(lmbda_4=lmbda, lmbda_g=0.0, sigma_u=sigma_u, tau=tau)
        
        nuts_kernel = infer.NUTS(target_potential, target_accept_prob=0.8)
        mcmc_runner = infer.MCMC(nuts_kernel,
                                  num_samples=n_samples,
                                  warmup_steps=n_warmup,
                                  num_chains=1
                                  )
        mcmc_runner.run()

        # 1. Capture raw coordinate samples
        raw_samples = mcmc_runner.get_samples()
        chain_samples.append(raw_samples)
        chain_diagnostics.append(mcmc_runner.diagnostics())
        
        # 2. Reconstruct D_tilde sample-by-sample manually with zero-gradient memory safety
        print("   Reconstructing scale-normalized matrices...")
        with torch.no_grad():
            u_samples = raw_samples["u"] # Shape: (n_samples, N, K)
            D_chain = []
            
            for s_idx in range(u_samples.shape[0]):
                u = u_samples[s_idx]
                
                # Re-run just the geometric projection steps from model.py
                u_norm = torch.norm(u, p=2, dim=-1, keepdim=True)
                u_norm_stable = torch.clamp(u_norm, min=1e-7)
                x = (u / u_norm_stable) * torch.tanh(u_norm)
                
                from src.geometry import compute_distance_matrix
                D_raw = compute_distance_matrix(x)
                
                # Scale-normalization anchor calculation
                triu_indices = torch.triu_indices(N, N, offset=1)
                mean_D = torch.clamp(D_raw[triu_indices[0], triu_indices[1]].mean(), min=1e-6)
                D_tilde = D_raw / mean_D
                D_chain.append(D_tilde)
                
            chain_D_matrices.append(torch.stack(D_chain))

    # Combining posterior coordinates from both chains
    combined_u = torch.stack([cs["u"] for cs in chain_samples], dim=0)
    
    # Pool and flatten the scale-normalized matrices safely
    combined_D = torch.cat(chain_D_matrices, dim=0) 
    posterior_distance_matrices[lmbda] = combined_D

    # Calculate sampler health diagnostics safely
    total_divergences = 0
    for diag in chain_diagnostics:
        div_indices = diag.get("diverging", [])
        if isinstance(div_indices, (list, np.ndarray, torch.Tensor)):
            total_divergences += int(np.sum(div_indices))
        else:
            total_divergences += int(div_indices)

    # Convert combined_u to NumPy for ArviZ compatibility
    az_summary = az.summary(az.from_dict(posterior={"u": combined_u.cpu().numpy()}))

    sampler_health_records[lmbda] = { 
        "ess": az_summary["ess_bulk"].mean(),
        "rhat": az_summary["r_hat"].max(),
        "divergences": total_divergences
    }
    
    print(f" -> Lambda {lmbda} complete. Max Rhat: {sampler_health_records[lmbda]['rhat']:.4f}, Divergences: {total_divergences}")
    sweep_records = []

for lmbda in lambda_grid:
    print(f"Evaluating scale-normalized test diagnostics for lambda = {lmbda}...")
    
    # Extract the pre-saved (S, N, N) normalized matrix batch for this lambda
    D_samples = posterior_distance_matrices[lmbda]
    
    # Run our updated validation diagnostic suite
    run_metrics = evaluate_test_diagnostics(D_samples, test_quartets, epsilon=0.05, gamma=0.05)
    
    # Extract the sampler health stats we compiled in the previous cell
    health = sampler_health_records[lmbda]
    
    # Append everything into our clean summary list
    sweep_records.append({
        "lambda": lmbda,
        "median_violation": run_metrics["hard_violations"]["median"],
        "quantile_95_violation": run_metrics["hard_violations"]["quantile_95"], # Addendum Target
        "tree_consistency_rate": run_metrics["tree_consistency_rate"],
        "star_like_fraction": run_metrics["quartet_gap"]["unresolved_fraction"], # Monitors star failure mode
        "median_pairwise_distance": run_metrics["distance_scale"]["median"],     # Should consistently be ~1.0
        "ess": health["ess"],
        "rhat": health["rhat"],
        "divergences": health["divergences"]
    })


LAUNCHING HYPERBOLIC PRIOR SWEEP (taxa : 50, Penalty budget: 3000)

 Sampling active landscape for lambda = 0.0
 Executing markov chain 1 / 2..


Sample: 100%|██████████| 600/600 [00:06, 98.92it/s, step size=4.21e-01, acc. prob=0.880] 


   Reconstructing scale-normalized matrices...
 Executing markov chain 2 / 2..


Sample: 100%|██████████| 600/600 [00:05, 104.31it/s, step size=4.27e-01, acc. prob=0.864]


   Reconstructing scale-normalized matrices...
 -> Lambda 0.0 complete. Max Rhat: 1.0200, Divergences: 0

 Sampling active landscape for lambda = 0.01
 Executing markov chain 1 / 2..


...

Sample: 100%|██████████| 600/600 [00:19, 30.47it/s, step size=4.36e-01, acc. prob=0.854]


   Reconstructing scale-normalized matrices...
 Executing markov chain 2 / 2..


Sample: 100%|██████████| 600/600 [00:22, 27.11it/s, step size=4.11e-01, acc. prob=0.884]


   Reconstructing scale-normalized matrices...
 -> Lambda 25.0 complete. Max Rhat: 1.0300, Divergences: 0
Evaluating scale-normalized test diagnostics for lambda = 0.0...


NameError: name 'N' is not defined

Evaluating scale-normalized test diagnostics for lambda = 0.0...


NameError: name 'N' is not defined

In [ ]:
# Convert to DataFrame for visualization and tracking
df_results = pd.DataFrame(sweep_records)

# Create results directory if it doesn't exist and save out a permanent CSV
import os
os.makedirs("./results", exist_ok=True)
df_results.to_csv("./results/lambda_sweep_diagnostics.csv", index=False)

print("\n### SWEEP METRICS COMPILED ###")
print(df_results[["lambda", "quantile_95_violation", "tree_consistency_rate", "star_like_fraction", "median_pairwise_distance"]])


## Plots

### checking the epsilon threshold for tree consistency across different values of lambda to make the right choice

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 11, 'axes.labelsize': 12, 'axes.titlesize': 13,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'figure.titlesize': 14
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))
fig.suptitle("Scale-Normalized Hyperbolic Prior Sweep Diagnostics", weight='bold', y=0.98)

# -------------------------------------------------------------------------
# PANEL 1: Structural Tree Alignment Metrics
# -------------------------------------------------------------------------
color_95 = '#d95f02'
color_rate = '#1f77b4'

# Line 1: 95th Percentile Hard Violation (Targeting 0.0)
ax1.plot(df_results["lambda"], df_results["quantile_95_violation"], 
         marker='o', linewidth=2.5, color=color_95, label="95th %ile Violation")
ax1.set_xlabel(r"Regularization Strength ($\lambda$)", weight='semibold')
ax1.set_ylabel("Normalized Violation Magnitude ($q_{0.95}(v_q)$)", color=color_95, weight='semibold')
ax1.tick_params(axis='y', labelcolor=color_95)
ax1.set_xscale('symlog', linthresh=0.01) 

# Twin axis for the Consistency Rate Percentage
ax1_twin = ax1.twinx()
ax1_twin.plot(df_results["lambda"], df_results["tree_consistency_rate"] * 100, 
              marker='s', linewidth=2, linestyle='--', color=color_rate, label="Consistency %")
ax1_twin.set_ylabel("Approx. Tree-Consistency Rate (%)", color=color_rate, weight='semibold')
ax1_twin.tick_params(axis='y', labelcolor=color_rate)
ax1_twin.grid(False) 

ax1.set_title("Structural Tree Alignment\n(Lower Violations & Higher Consistency = Better)", pad=10)

# -------------------------------------------------------------------------
# PANEL 2: Latent Geometry Safeguards & Sampler Health
# -------------------------------------------------------------------------
color_star = '#7570b3'
color_div = '#e7298a'

# Line 2: Star-like Failure Mode Fraction
ax2.plot(df_results["lambda"], df_results["star_like_fraction"] * 100, 
         marker='^', linewidth=2.5, color=color_star, label="Star-like Fraction")
ax2.set_xlabel(r"Regularization Strength ($\lambda$)", weight='semibold')
ax2.set_ylabel("Unresolved Star-tree Fraction (%)", color=color_star, weight='semibold')
ax2.tick_params(axis='y', labelcolor=color_star)
ax2.set_xscale('symlog', linthresh=0.01)

# Twin axis for Sampler Health (MCMC Divergences)
ax2_twin = ax2.twinx()
ax2_twin.bar(df_results["lambda"], df_results["divergences"], alpha=0.3, color=color_div, 
             width=[0.005 if l == 0 else l*0.3 for l in df_results["lambda"]], label="Divergences")
ax2_twin.set_ylabel("MCMC Divergence Count", color=color_div, weight='semibold')
ax2_twin.tick_params(axis='y', labelcolor=color_div)
ax2_twin.grid(False)

ax2.set_title("Geometry Safeguards & Sampler Health\n(Watch for Star-Tree Spikes or Divergence Explosions)", pad=10)

plt.tight_layout()
plt.savefig("./results/normalized_lambda_sweep_curves.png", dpi=300, bbox_inches='tight')
plt.show()

### Current experiment interpretation:

Even though we increased the penalty strength to value of 100, the tree consistency rate increases but overall these values are too low because training quuartet budget = 3000 is too small for total quartets 2,30,300.

Overall, for $\lambda$ = 0, only 12% of the test quartets, satisfy the tree condition. For  $\lambda$ = 100, this number increased to 20.1%.

The coverage of quartets currently is only 1.3% and thus represents we are under-sampling during training.
so, our second experiment would be to increase the quartet budget!